# 15-Minute Wind Resampling

This notebook resamples 10-minute wind data to 15-minute resolution.

In [1]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path

### Load and Combine Wind Files

In [2]:
# find all Oulu wind CSV files (10-min raw data)
# NOTE: cwd = notebook directory (WindDirection&Speed folder)
wind_dir = Path('.')
wind_files = sorted(wind_dir.glob('Oulu Vihreäsaari harbour_*.csv'))

print(f'Found {len(wind_files)} wind files:')
for f in wind_files:
    print(f'  {f.name}')

Found 9 wind files:
  Oulu Vihreäsaari harbour_ 1.1.2023 - 30.4.2023.csv
  Oulu Vihreäsaari harbour_ 1.1.2024 - 30.4.2024.csv
  Oulu Vihreäsaari harbour_ 1.1.2025 - 30.4.2025.csv
  Oulu Vihreäsaari harbour_ 1.5.2023 - 31.8.2023.csv
  Oulu Vihreäsaari harbour_ 1.5.2024 - 31.8.2024.csv
  Oulu Vihreäsaari harbour_ 1.5.2025 - 31.8.2025.csv
  Oulu Vihreäsaari harbour_ 1.9.2023 - 31.12.2023.csv
  Oulu Vihreäsaari harbour_ 1.9.2024 - 31.12.2024.csv
  Oulu Vihreäsaari harbour_ 1.9.2025 - 31.12.2025.csv


In [3]:
# read and combine all wind files into one DataFrame
wind_list = []
for f in wind_files:
    w = pd.read_csv(f)
    w['datetime'] = pd.to_datetime(
        w['Year'].astype(str) + '-' +
        w['Month'].astype(str).str.zfill(2) + '-' +
        w['Day'].astype(str).str.zfill(2) + ' ' +
        w['Time [Local time]'].astype(str)
    )
    w['datetime'] = w['datetime'].dt.tz_localize('Europe/Helsinki', ambiguous='NaT', nonexistent='NaT')
    w = w[['datetime', 'Wind direction mean [°]', 'Wind speed mean [m/s]']].rename(
        columns={
            'Wind direction mean [°]': 'wind_direction_deg',
            'Wind speed mean [m/s]': 'wind_speed_ms'
        }
    )
    # force numeric conversion to avoid string dtype issues
    w['wind_direction_deg'] = pd.to_numeric(w['wind_direction_deg'], errors='coerce')
    w['wind_speed_ms'] = pd.to_numeric(w['wind_speed_ms'], errors='coerce')
    wind_list.append(w)

wind_df = pd.concat(wind_list, ignore_index=True)
wind_df = wind_df.sort_values('datetime').drop_duplicates(subset=['datetime']).reset_index(drop=True)

print('Wind combined shape:', wind_df.shape)
print('Date range:', wind_df['datetime'].min(), '->', wind_df['datetime'].max())
wind_df.head(3)

Wind combined shape: (157789, 3)
Date range: 2023-01-01 00:00:00+02:00 -> 2025-12-31 23:50:00+02:00


,datetime,wind_direction_deg,wind_speed_ms
0,2023-01-01 00:00:00+02:00,215.3,8.7
1,2023-01-01 00:10:00+02:00,215.4,8.2
2,2023-01-01 00:20:00+02:00,213.1,9.0


### Resample Wind to 15-Minute
#
 Wind direction is circular (0 and 360 are the same).
 It need convert to sin/cos first, resample those, then reconstruct the angle.

In [4]:
# convert wind direction degrees to sin/cos BEFORE resampling
rad = np.deg2rad(wind_df['wind_direction_deg'].astype(float))
wind_df['wind_dir_sin'] = np.sin(rad)
wind_df['wind_dir_cos'] = np.cos(rad)

# set datetime as index
wind_10min = wind_df.set_index('datetime')

# resample wind speed using mean (simple average)
wind_speed_15min = wind_10min[['wind_speed_ms']].resample('15min').mean()

# resample sin/cos using mean (circular average)
wind_sin_15min = wind_10min[['wind_dir_sin']].resample('15min').mean()
wind_cos_15min = wind_10min[['wind_dir_cos']].resample('15min').mean()

# reconstruct wind direction degrees from averaged sin/cos
wind_dir_15min = np.rad2deg(
    np.arctan2(wind_sin_15min['wind_dir_sin'], wind_cos_15min['wind_dir_cos'])
) % 360

# combine into one DataFrame
wind_15min = pd.DataFrame({
    'wind_speed_ms': wind_speed_15min['wind_speed_ms'],
    'wind_direction_deg': wind_dir_15min,
    'wind_dir_sin': wind_sin_15min['wind_dir_sin'],
    'wind_dir_cos': wind_cos_15min['wind_dir_cos'],
})

print('Wind 15-min shape:', wind_15min.shape)
print('NaN count:', wind_15min.isna().sum().sum())
wind_15min.head(6)

Wind 15-min shape: (105216, 4)
NaN count: 640


,wind_speed_ms,wind_direction_deg,wind_dir_sin,wind_dir_cos
datetime,,,,
2023-01-01 00:00:00+02:00,8.45,215.35,-0.578569,-0.815633
2023-01-01 00:15:00+02:00,9.00,213.10,-0.546102,-0.837719
2023-01-01 00:30:00+02:00,8.55,211.25,-0.518650,-0.854708
2023-01-01 00:45:00+02:00,8.10,209.70,-0.495459,-0.868632
2023-01-01 01:00:00+02:00,8.10,211.85,-0.527451,-0.849036
2023-01-01 01:15:00+02:00,7.50,217.10,-0.603208,-0.797584


### Save Resampled 15-Minute Weather Files

In [5]:
# save wind 15-min (cwd = notebook folder, so relative path works)
wind_15min.to_csv('wind_15min.csv', index=True)
print('Saved: wind_15min.csv')

Saved: wind_15min.csv


In [6]:
print('\n=== Summary ===')

print(f'Wind:        {len(wind_15min)} rows, {wind_15min.index.min()} to {wind_15min.index.max()}')


=== Summary ===
Wind:        105216 rows, 2023-01-01 00:00:00+02:00 to 2025-12-31 23:45:00+02:00
